# Run-book 145-x5 — MPC keeps a granule exactly at `max_cloudcover`

**What this proves:** gate 4 for #145 X5 (spec 32 Amendment A1). `max_cloudcover` now means "at
most" on MPC: the STAC query sends `lte`, not `lt`. On the tutorial ROI and window, fsd with
`max_cloudcover` set to the lowest cloud cover `c` returns that granule (Step 2). Step 3 sends the
same raw request to MPC with `lte` and with `lt`: the server's `lte` result holds the granule at
`c` and nothing above it, and `lt` drops it. Together they show MPC's server applies `lte` at
exactly the threshold. The AML discovery path (`discover_shard_rows`) sends the same query; the
test `tests/test_mpc_cloudcover_lte.py` pins it, so it has no step here.

**Claude does not run this** (spec 24). You run it cell by cell and paste back the **Summary**
cell's output, or a failing cell's message. Every check is a plain `assert` you can read: the
text above a check cell says what PASS means, and the `assert` says the same thing in code.

**Needs:** network access to the Planetary Computer STAC API (anonymous; no key). Nothing is
downloaded: three searches, a few seconds each (estimate). Disk: one small JSON file.

<!-- Rules for writing one (spec 24 amendment A1):
  - call fsd's public functions directly, keyword arguments spelled out;
  - small helpers only (~15 lines), defined just above their first use, with a docstring;
  - no environment variables, no try/except around a whole step, no result-file plumbing;
  - every step reads what it needs from disk + Settings, never from an earlier step's variables;
  - a plan cell (counts + GB, no transfer) before any step that moves more than ~1 GB;
  - a deliberate kill runs in a child process (os._exit would kill this kernel). -->

## Before you start

**Kernel — which Python runs the cells.** Use **`.venv`**: the Python at `fsd/.venv/bin/python`
in the **main** checkout (Python 3.11). It is the only environment with fsd's dependencies.
- Open VS Code on the **`fsd/` folder** (the main checkout) — even when this notebook lives in a
  worktree under `fsd/.claude/worktrees/<name>/`. Then **Select Kernel** (top right) →
  **Python Environments** → **`.venv`**.
- Not listed (e.g. VS Code was opened on the worktree folder, which has no `.venv` of its own)?
  **Select Another Kernel… → Python Environments → Enter interpreter path…** (wording varies by
  VS Code version) and give the main checkout's `fsd/.venv/bin/python`.
- The **Setup** cell below checks this and stops with a message if the kernel is wrong.

**Code — which `fsd` gets imported.** fsd is installed from the *main* checkout, so inside a
worktree a plain `import fsd` would quietly run `main`'s code, not the branch under test. The
Setup cell puts **this notebook's own** `src/` first on Python's import path and checks it worked.
(This is what the old run-books did with `export PYTHONPATH=src` — the same thing, invisibly.)

**Stop / resume.** Interrupt the kernel (■) to stop. Re-running a cell resumes: downloads skip
files that are already complete. After a kernel restart, run **Setup** and **Settings** again,
then continue at any step — every step reads what it needs from disk.

**When you are done.** Paste back what the **Summary** cell says to paste (plus any failing cell's
message). Then **Restart & Clear All Outputs** and save: this file is tracked in git, and a test
refuses saved outputs.

In [ ]:
# Setup -- which Python, and which fsd. Nothing to change here.
import sys
from pathlib import Path

# 1. Which Python? It must be fsd's own virtual environment, fsd/.venv.
VENV = Path(sys.prefix)
assert VENV.name == ".venv" and (VENV.parent / "pyproject.toml").exists(), (
    f"Wrong kernel: {sys.prefix} is not fsd's .venv. Pick the '.venv' kernel (see above).")
MAIN_CHECKOUT = VENV.parent              # the fsd checkout that owns the venv

# 2. Which fsd? The one next to this notebook: <checkout>/runbooks/<this> -> <checkout>/src.
#    Jupyter and VS Code start the kernel in the notebook's own folder.
CHECKOUT = Path.cwd().parent
assert Path.cwd().name == "runbooks" and (CHECKOUT / "src" / "fsd").is_dir(), (
    f"The kernel started in {Path.cwd()}, not in this notebook's runbooks/ folder.")
if str(CHECKOUT / "src") not in sys.path:
    sys.path.insert(0, str(CHECKOUT / "src"))
import fsd

assert Path(fsd.__file__).is_relative_to(CHECKOUT / "src"), (
    f"fsd was already imported from {fsd.__file__}. Restart the kernel, then run this cell first.")
print("Python :", sys.executable)
print("fsd    :", fsd.__file__)

## Settings

Every input this run-book uses. Change them here and nowhere else.

In [ ]:
# Settings
import datetime

OUT = MAIN_CHECKOUT / "tests" / "outputs" / "145_x5_mpc_cloudcover"   # gitignored
ROI = CHECKOUT / "notebooks" / "shapefiles" / "s2grid=4772924.geojson"   # inside T33UWP
STARTDATE = datetime.datetime(2018, 4, 1)    # the tutorial fixture's window
ENDDATE = datetime.datetime(2018, 10, 1)
# The fixture's catalog records 0.066284 as the lowest cloud cover in this window
# (S2B_MSIL2A_20180928T100019_R122_T33UWP). Step 1 reads the live value instead, in case MPC
# has reprocessed since.
FIXTURE_LOWEST = 0.066284

## Step 1 — find the lowest cloud cover in the window

**What it does:** searches MPC with no cloud filter (`fsd.sources.mpc.query_catalog`) and writes
the granule with the lowest `cloud_cover`, and that value `c`, to `OUT/lowest.json`.
**PASS if:** at least one granule comes back and `c` is above 0 (a granule at exactly 0 would
still test the boundary, but every value in this window is a six-decimal float above 0).
**Time / disk:** seconds; one small file.

In [ ]:
# Step 1 -- an unfiltered search; keep the lowest-cloud granule.
import json
from fsd.sources import mpc

OUT.mkdir(parents=True, exist_ok=True)
found = mpc.query_catalog(str(ROI), STARTDATE, ENDDATE, max_cloudcover=None)
lowest = found.loc[found["cloud_cover"].idxmin()]
(OUT / "lowest.json").write_text(json.dumps(
    {"id": lowest["id"], "cloud_cover": float(lowest["cloud_cover"]), "n_granules": len(found)}))
print(len(found), "granules; lowest:", lowest["id"], float(lowest["cloud_cover"]))

In [ ]:
# Step 1 -- check it.
low = json.loads((OUT / "lowest.json").read_text())
assert low["n_granules"] > 0, "expected granules in the window, found none"
assert low["cloud_cover"] > 0, f"expected a lowest cloud cover above 0, found {low['cloud_cover']}"
print("PASS", low, "| fixture recorded", FIXTURE_LOWEST)

## Step 2 — `max_cloudcover = c` keeps that granule (the change)

**What it does:** the same search with `max_cloudcover=c`.
**PASS if:** the lowest granule from Step 1 is in the result. On `main` before X5 (`lt`) it is not.
**Time / disk:** seconds; nothing written.

In [ ]:
# Step 2 -- search at exactly c, through fsd.
low = json.loads((OUT / "lowest.json").read_text())
at_c = mpc.query_catalog(str(ROI), STARTDATE, ENDDATE, max_cloudcover=low["cloud_cover"])
step2_ids = sorted(at_c["id"])
print(len(step2_ids), "granules at max_cloudcover =", low["cloud_cover"])
assert low["id"] in step2_ids, f"expected {low['id']} at max_cloudcover={low['cloud_cover']}, got {step2_ids}"
print("PASS")

## Step 3 — MPC's server applies `lte` at the threshold, and `lt` excludes it

**What it does:** sends the same raw `pystac_client` search to MPC twice, at `c`: once with
`query={"eo:cloud_cover": {"lte": c}}` (what fsd sends now) and once with `{"lt": c}` (what it sent
before X5). It goes around fsd on purpose: fsd filters again locally with `<=`, so Step 2 alone
cannot tell a server that applies `lte` from one that ignores the filter.
**PASS if:** the `lte` result is not empty, holds an item with cloud cover exactly `c`, and every
item in it has cloud cover `<= c` (the server filtered); and the `lt` result holds no item at `c`.
**Time / disk:** seconds; nothing written.

In [ ]:
# Step 3 -- the same raw search with lte and with lt.
import geopandas as gpd
import pystac_client
import shapely
from fsd import config


def raw_covers(op, value):
    """Cloud covers of the items MPC returns for `{"eo:cloud_cover": {op: value}}`."""
    geom = shapely.unary_union(gpd.read_file(ROI).to_crs("EPSG:4326")["geometry"])
    items = pystac_client.Client.open(config.MPC_STAC_URL).search(
        collections=[config.SATELLITE_S2L2A], datetime=[STARTDATE, ENDDATE], intersects=geom,
        query={"eo:cloud_cover": {op: value}}, limit=200).items()
    return sorted(it.properties["eo:cloud_cover"] for it in items)


low = json.loads((OUT / "lowest.json").read_text())
c = low["cloud_cover"]
lte, lt = raw_covers("lte", c), raw_covers("lt", c)
print(len(lte), "items with lte:", lte, "|", len(lt), "items with lt:", lt)
assert lte and c in lte, f"expected an item at exactly {c} with lte, got {lte}"
assert all(v <= c for v in lte), f"expected every lte item <= {c} (server filtered), got {lte}"
assert c not in lt, f"expected no item at {c} with lt, got {lt}"
print("PASS")

## Summary — paste this output back

Re-reads Step 1's file and repeats Steps 2 and 3, so it works after a restart (run Setup, Settings and the Step 3 cell first: it defines `raw_covers`).

In [ ]:
# Summary -- repeats Steps 2 and 3 from disk + Settings (needs Step 3's raw_covers).
import json
from fsd.sources import mpc

low = json.loads((OUT / "lowest.json").read_text())
c = low["cloud_cover"]
at_c = mpc.query_catalog(str(ROI), STARTDATE, ENDDATE, max_cloudcover=c)
lte, lt = raw_covers("lte", c), raw_covers("lt", c)   # raw_covers: defined in Step 3
summary = {
    "fsd": fsd.__file__,
    "lowest": low,
    "fixture_lowest": FIXTURE_LOWEST,
    "fsd_keeps_lowest": low["id"] in set(at_c["id"]),
    "raw_lte_n": len(lte),
    "raw_lte_has_c": c in lte,
    "raw_lte_all_le_c": all(v <= c for v in lte),
    "raw_lt_n": len(lt),
    "raw_lt_has_c": c in lt,
}
print(json.dumps(summary, indent=2, default=str))